In [8]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_path = "../data/"


df_airports = pd.read_csv(data_path + "airportgroups.csv")
print(f"Airport Groups: {df_airports.shape}")

df_hist = pd.read_csv(data_path + "historical_flights.csv")
print(f"Historical Flights: {df_hist.shape}")

df_sched = pd.read_csv(data_path + "schedule_oct2025.csv")
print(f"Schedule Oct 2025: {df_sched.shape}")



print("\nDataset Summary:")
datasets = {
    'Airport Groups': df_airports,
    'Historical Flights': df_hist,
    'Schedule Oct 2025': df_sched,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Airport Groups: (21, 2)
Historical Flights: (410442, 11)
Schedule Oct 2025: (3984, 8)

Dataset Summary:
Airport Groups: 21 rows × 2 columns
Historical Flights: 410,442 rows × 11 columns
Schedule Oct 2025: 3,984 rows × 8 columns


In [9]:
# ====================================================================================================
# TEMPORAL FEATURE ENGINEERING
# ====================================================================================================

def create_temporal_features(df):
    """Create comprehensive temporal features from departure and arrival times"""
    print("Creating temporal features from departure (STD) and arrival (STA) times...")
    
    # Parse departure and arrival times
    df['std_parsed'] = pd.to_datetime(df['std'])
    df['sta_parsed'] = pd.to_datetime(df['sta']) if 'sta' in df.columns else None
    
    # DEPARTURE TIME FEATURES
    dt_dep = df['std_parsed'].dt
    
    # Basic departure temporal features
    df['dep_hour'] = dt_dep.hour
    df['dep_day_of_week'] = dt_dep.dayofweek
    df['dep_month'] = dt_dep.month
    df['dep_day_of_month'] = dt_dep.day
    df['dep_week_of_year'] = dt_dep.isocalendar().week
    df['dep_quarter'] = dt_dep.quarter
    df['dep_year'] = dt_dep.year
    
    # Departure day of week binary features
    df['dep_is_monday'] = (dt_dep.dayofweek == 0).astype(int)
    df['dep_is_tuesday'] = (dt_dep.dayofweek == 1).astype(int)
    df['dep_is_wednesday'] = (dt_dep.dayofweek == 2).astype(int)
    df['dep_is_thursday'] = (dt_dep.dayofweek == 3).astype(int)
    df['dep_is_friday'] = (dt_dep.dayofweek == 4).astype(int)
    df['dep_is_saturday'] = (dt_dep.dayofweek == 5).astype(int)
    df['dep_is_sunday'] = (dt_dep.dayofweek == 6).astype(int)
    
    # Departure time categories
    df['dep_is_weekend'] = (dt_dep.dayofweek >= 5).astype(int)
    df['dep_hour_category'] = pd.cut(df['dep_hour'], 
                                    bins=[0, 6, 9, 12, 17, 20, 24], 
                                    labels=['Night', 'Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Late_Evening'],
                                    include_lowest=True)
    
    # Departure rush hour indicators
    df['dep_morning_rush'] = ((df['dep_hour'] >= 5) & (df['dep_hour'] <= 7)).astype(int)
    df['dep_evening_rush'] = ((df['dep_hour'] >= 17) & (df['dep_hour'] <= 19)).astype(int)
    df['dep_business_hours'] = ((df['dep_hour'] >= 7) & (df['dep_hour'] <= 18)).astype(int)
    
    # Departure cyclical features
    df['dep_hour_sin'] = np.sin(2 * np.pi * df['dep_hour'] / 24)
    df['dep_hour_cos'] = np.cos(2 * np.pi * df['dep_hour'] / 24)
    df['dep_day_sin'] = np.sin(2 * np.pi * df['dep_day_of_week'] / 7)
    df['dep_day_cos'] = np.cos(2 * np.pi * df['dep_day_of_week'] / 7)
    df['dep_month_sin'] = np.sin(2 * np.pi * df['dep_month'] / 12)
    df['dep_month_cos'] = np.cos(2 * np.pi * df['dep_month'] / 12)
    
    # ARRIVAL TIME FEATURES (if STA exists)
    if df['sta_parsed'] is not None and pd.notna(df['sta_parsed']).any():
        dt_arr = df['sta_parsed'].dt
        
        # Basic arrival temporal features
        df['arr_hour'] = dt_arr.hour
        df['arr_day_of_week'] = dt_arr.dayofweek
        df['arr_month'] = dt_arr.month
        
        # Arrival day of week binary features
        df['arr_is_monday'] = (dt_arr.dayofweek == 0).astype(int)
        df['arr_is_tuesday'] = (dt_arr.dayofweek == 1).astype(int)
        df['arr_is_wednesday'] = (dt_arr.dayofweek == 2).astype(int)
        df['arr_is_thursday'] = (dt_arr.dayofweek == 3).astype(int)
        df['arr_is_friday'] = (dt_arr.dayofweek == 4).astype(int)
        df['arr_is_saturday'] = (dt_arr.dayofweek == 5).astype(int)
        df['arr_is_sunday'] = (dt_arr.dayofweek == 6).astype(int)
        
        # Arrival time categories
        df['arr_is_weekend'] = (dt_arr.dayofweek >= 5).astype(int)
        df['arr_hour_category'] = pd.cut(df['arr_hour'], 
                                        bins=[0, 6, 9, 12, 17, 20, 24], 
                                        labels=['Night', 'Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Late_Evening'],
                                        include_lowest=True)
        
        # Arrival rush hour indicators
        df['arr_morning_rush'] = ((df['arr_hour'] >= 5) & (df['arr_hour'] <= 7)).astype(int)
        df['arr_evening_rush'] = ((df['arr_hour'] >= 17) & (df['arr_hour'] <= 19)).astype(int)
        df['arr_business_hours'] = ((df['arr_hour'] >= 7) & (df['arr_hour'] <= 18)).astype(int)
        
        # Arrival cyclical features
        df['arr_hour_sin'] = np.sin(2 * np.pi * df['arr_hour'] / 24)
        df['arr_hour_cos'] = np.cos(2 * np.pi * df['arr_hour'] / 24)
        df['arr_day_sin'] = np.sin(2 * np.pi * df['arr_day_of_week'] / 7)
        df['arr_day_cos'] = np.cos(2 * np.pi * df['arr_day_of_week'] / 7)
        df['arr_month_sin'] = np.sin(2 * np.pi * df['arr_month'] / 12)
        df['arr_month_cos'] = np.cos(2 * np.pi * df['arr_month'] / 12)
        
        # FLIGHT DURATION & CROSS-TIME FEATURES
        df['scheduled_duration'] = (df['sta_parsed'] - df['std_parsed']).dt.total_seconds() / 3600  # hours
        df['overnight_flight'] = (df['arr_day_of_week'] != df['dep_day_of_week']).astype(int)
        
        # Time span features
        df['crosses_midnight'] = (df['arr_hour'] < df['dep_hour']).astype(int)
        df['peak_to_peak'] = ((df['dep_morning_rush'] | df['dep_evening_rush']) & 
                             (df['arr_morning_rush'] | df['arr_evening_rush'])).astype(int)
    
    # SHARED TEMPORAL FEATURES (using departure time as base)
    # Season mapping
    season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
                  3: 'Spring', 4: 'Spring', 5: 'Spring',
                  6: 'Summer', 7: 'Summer', 8: 'Summer',
                  9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}
    df['season'] = df['dep_month'].map(season_map)
    
    # Legacy features for compatibility (using departure time)
    df['hour'] = df['dep_hour']
    df['day_of_week'] = df['dep_day_of_week'] 
    df['month'] = df['dep_month']
    
    temporal_feature_count = len([col for col in df.columns if any(x in col.lower() for x in ['dep_', 'arr_', 'hour', 'day', 'month', 'season', 'rush', 'weekend', 'business', 'overnight', 'duration'])])
    print(f"Created {temporal_feature_count} temporal features (departure + arrival + flight timing)")
    return df

# Apply temporal features to historical data
print("TEMPORAL FEATURE ENGINEERING")
print("=" * 50)
df_hist_enhanced = df_hist.copy()
df_hist_enhanced = create_temporal_features(df_hist_enhanced)

# Show temporal features created
temporal_cols = [col for col in df_hist_enhanced.columns if any(x in col.lower() for x in ['hour', 'day', 'month', 'season', 'rush', 'weekend', 'business'])]
print(f"Temporal features created: {temporal_cols}")

TEMPORAL FEATURE ENGINEERING
Creating temporal features from departure (STD) and arrival (STA) times...
Created 56 temporal features (departure + arrival + flight timing)
Temporal features created: ['dep_hour', 'dep_day_of_week', 'dep_month', 'dep_day_of_month', 'dep_is_monday', 'dep_is_tuesday', 'dep_is_wednesday', 'dep_is_thursday', 'dep_is_friday', 'dep_is_saturday', 'dep_is_sunday', 'dep_is_weekend', 'dep_hour_category', 'dep_morning_rush', 'dep_evening_rush', 'dep_business_hours', 'dep_hour_sin', 'dep_hour_cos', 'dep_day_sin', 'dep_day_cos', 'dep_month_sin', 'dep_month_cos', 'arr_hour', 'arr_day_of_week', 'arr_month', 'arr_is_monday', 'arr_is_tuesday', 'arr_is_wednesday', 'arr_is_thursday', 'arr_is_friday', 'arr_is_saturday', 'arr_is_sunday', 'arr_is_weekend', 'arr_hour_category', 'arr_morning_rush', 'arr_evening_rush', 'arr_business_hours', 'arr_hour_sin', 'arr_hour_cos', 'arr_day_sin', 'arr_day_cos', 'arr_month_sin', 'arr_month_cos', 'season', 'hour', 'day_of_week', 'month']


In [ ]:
# ====================================================================================================
# FLIGHT DENSITY & OPERATIONAL FEATURES
# ====================================================================================================

def create_density_features(df):
    """Create flight density and operational load features"""
    print("Creating flight density features...")
    
    # Sort by datetime for window operations
    df = df.sort_values('std_parsed').reset_index(drop=True)
    
    # Airport group flight counts by time windows
    df['date'] = df['std_parsed'].dt.date
    df['hour_slot'] = df['std_parsed'].dt.floor('H')
    
    # Flights per hour per airport group
    hourly_counts = df.groupby(['dep_airport_group', 'hour_slot']).size().reset_index()
    hourly_counts.columns = ['dep_airport_group', 'hour_slot', 'flights_this_hour']
    df = df.merge(hourly_counts, on=['dep_airport_group', 'hour_slot'], how='left')
    
    # Flights per day per airport group
    daily_counts = df.groupby(['dep_airport_group', 'date']).size().reset_index()
    daily_counts.columns = ['dep_airport_group', 'date', 'flights_this_day']
    df = df.merge(daily_counts, on=['dep_airport_group', 'date'], how='left')
    
    # System-wide load
    system_hourly = df.groupby('hour_slot').size().reset_index()
    system_hourly.columns = ['hour_slot', 'system_flights_this_hour']
    df = df.merge(system_hourly, on='hour_slot', how='left')
    
    # Relative load (airport group vs system)
    df['relative_load'] = df['flights_this_hour'] / (df['system_flights_this_hour'] + 1)
    
    # Service type diversity
    service_counts = df.groupby(['dep_airport_group', 'hour_slot', 'service_type']).size().unstack(fill_value=0)
    service_counts.columns = [f'service_{col}_count' for col in service_counts.columns]
    service_counts = service_counts.reset_index()
    df = df.merge(service_counts, on=['dep_airport_group', 'hour_slot'], how='left')
    
    # Fill NaN values
    service_cols = [col for col in df.columns if col.startswith('service_') and col.endswith('_count')]
    df[service_cols] = df[service_cols].fillna('J')
    
    print(f"Created density features: flights_this_hour, flights_this_day, system_flights_this_hour, relative_load")
    print(f"Created service type features: {len(service_cols)} service type counts")
    
    return df

# Apply density features
print("\nFLIGHT DENSITY FEATURES")
print("=" * 50)
df_hist_enhanced = create_density_features(df_hist_enhanced)


STEP 2: FLIGHT DENSITY FEATURES
Creating flight density features...
Created density features: flights_this_hour, flights_this_day, system_flights_this_hour, relative_load
Created service type features: 16 service type counts


In [ ]:
# ====================================================================================================
# DELAY & OPERATIONAL RELIABILITY FEATURES
# ====================================================================================================

def create_operational_features(df):
    """Create operational reliability and delay-based features"""
    print("Creating operational reliability features...")
    
    # Calculate delays if actual times exist
    if 'atd' in df.columns and pd.notna(df['atd']).any():
        df['atd_parsed'] = pd.to_datetime(df['atd'])
        df['departure_delay'] = (df['atd_parsed'] - df['std_parsed']).dt.total_seconds() / 60  # minutes
        
        # Delay categories
        df['delay_category'] = pd.cut(df['departure_delay'], 
                                    bins=[-np.inf, -5, 15, 60, np.inf],
                                    labels=['Early', 'On_Time', 'Delayed', 'Severely_Delayed'])
        
        # On-time performance by airport group
        df['is_on_time'] = (df['departure_delay'].abs() <= 15).astype(int)
        
    # Cancellation rates
    if 'cancelled' in df.columns:
        df['is_cancelled'] = df['cancelled'].fillna(0).astype(int)
    else:
        df['is_cancelled'] = 0
    
    # Flight duration (scheduled)
    if 'sta' in df.columns:
        df['sta_parsed'] = pd.to_datetime(df['sta'])
        df['scheduled_duration'] = (df['sta_parsed'] - df['std_parsed']).dt.total_seconds() / 3600  # hours
        
        # Duration categories
        df['duration_category'] = pd.cut(df['scheduled_duration'],
                                       bins=[0, 1, 2, 4, 8, np.inf],
                                       labels=['Short', 'Medium', 'Long', 'Very_Long', 'Ultra_Long'])
    
    print("Created operational features: delays, on-time performance, cancellations, flight duration")
    return df

# Apply operational features
print("\nOPERATIONAL RELIABILITY FEATURES")
print("=" * 50)
df_hist_enhanced = create_operational_features(df_hist_enhanced)


STEP 3: OPERATIONAL RELIABILITY FEATURES
Creating operational reliability features...
Created operational features: delays, on-time performance, cancellations, flight duration
